# Experiment — can a model guess which study a sample came from?

Hand the model **one stool sample's microbial composition** and nothing else —
no disease label, no country, no sequencing metadata — and ask it to name the
**study** that sample was collected in. 267 studies are on the table; always
naming the largest one scores about 4%.

A model that beats that by a wide margin is reading a *lab fingerprint*: the
DNA-extraction kit, the PCR primers, the sequencer, the bench protocol. Those
vary from study to study and each leaves a mark on the measured composition.
That fingerprint is exactly what every downstream disease comparison has to
work around, so it is worth seeing how strong it is on its own.

This notebook runs **only** that experiment. The full technical-variance
analysis (PERMANOVA, ordinations, depth vs richness) is in `04_variance.ipynb`;
the classifier itself is `run_project_cv` in `src/variance.py`.

## Setup

In [ ]:
import gc
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import cohorts as coh
from src import variance as var

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

with open(ROOT / "config" / "params.yaml") as f:
    params = yaml.safe_load(f)
SEED = params["seed"]
ccfg = params["variance"]["classifier"]

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
warnings.filterwarnings("ignore")

print(f"seed={SEED}  torch={torch.__version__}  device={DEVICE}")
print(f"classifier config: max_n={ccfg['max_n']:,}  min_project_size={ccfg['min_project_size']}  "
      f"folds={ccfg['cv_folds']}  epochs={ccfg['epochs']}  models=linear+mlp")

## 1 — Assemble the sample table

The QC-passed population, rebuilt with the same depth filter Phases 2 and 3 use
(cheap and deterministic, not persisted). `prepare_covariates` only normalises a
few metadata columns here — the experiment itself uses just `sample` and
`project`.

In [ ]:
harmonized = pd.read_parquet(INTERIM / "samples_harmonized.parquet")
sample_depth = pd.read_parquet(INTERIM / "sample_depth.parquet")

qc, _ = coh.apply_qc_filters(harmonized, sample_depth, params["qc"]["depth_threshold"])
qc = var.prepare_covariates(qc)
del harmonized
gc.collect()

print(f"QC-passed: {len(qc):,} samples across {qc['project'].nunique()} projects")

## 2 — Load each sample's composition

419 genus-level **CLR** features per sample — the only thing the model sees.

The population is first capped with a **project-stratified subsample** (every
study keeps its share of the rows), and only those rows are read from the CLR
Parquet — the full ~116k x 419 matrix never has to be resident.

In [ ]:
t0 = time.time()

# subsample the sample keys BEFORE touching the abundance file
pool = qc[["sample", "project"]].copy()
sub = var.project_stratified_subsample(pool, ccfg["max_n"], SEED)

ids, clr, _ = var.load_abundance_subset(PROCESSED / "abund_clr.parquet",
                                        set(sub["sample"]))
pos = {k: i for i, k in enumerate(ids)}
sub = sub[sub["sample"].isin(pos)].reset_index(drop=True)   # drop keys absent from the CLR file

row_idx = [pos[k] for k in sub["sample"]]
X = np.ascontiguousarray(clr[row_idx])
y = sub["project"].to_numpy()
samples = sub["sample"].to_numpy()
del clr, ids
gc.collect()

print(f"design matrix: {X.shape[0]:,} samples x {X.shape[1]} CLR features, "
      f"{pd.Series(y).nunique()} studies  ({time.time() - t0:.0f}s)")

## 3 — Run the experiment (5-fold cross-validated)

`run_project_cv` trains two models from scratch on each fold — `ProjectLinear`
(multinomial logistic regression, one `nn.Linear`) and `ProjectMLP`
(`419 → 256 → 128 → studies`) — with a `StandardScaler` fit on the training
rows only. Studies with fewer than `min_project_size` samples are dropped (too
few to stratify) and reported. Every score is next to the majority-class
baseline.

In [ ]:
t0 = time.time()
cv, summary = var.run_project_cv(
    X, y, SEED,
    n_splits=ccfg["cv_folds"], epochs=ccfg["epochs"], batch_size=ccfg["batch_size"],
    lr=ccfg["lr"], hidden=tuple(ccfg["hidden"]), min_project_size=ccfg["min_project_size"],
)
print(f"cross-validation done in {time.time() - t0:.0f}s\n")

print(f"studies kept:   {summary['n_classes']}   "
      f"(dropped {summary['n_projects_dropped']} studies < {summary['min_project_size']} samples "
      f"= {summary['n_samples_dropped']:,} samples)")
print(f"samples scored: {summary['n_samples']:,}")
print(f"majority-class baseline (always name the biggest study): {summary['majority_baseline']:.4f}\n")

table = (cv.groupby("model")[["accuracy", "balanced_accuracy", "top5_accuracy"]]
           .agg(["mean", "std"]).round(4))
print(table.to_string())

In [ ]:
lin = cv.loc[cv["model"] == "linear", "accuracy"].mean()
factor = lin / summary["majority_baseline"]
print(f"A one-layer linear model names the originating study {lin:.1%} of the time — "
      f"{factor:.0f}x the {summary['majority_baseline']:.1%} floor.")
print("The lab fingerprint is not subtle structure: it is the dominant, "
      "linearly separable axis of the data.")

## 4 — Look at the individual guesses

Cross-validation gives the headline number; this trains **one** linear model on
80% of the samples and lets it call the study for held-out samples it has never
seen, so the guesses can be inspected one by one.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# mirror the project-size filter run_project_cv applies internally
vc = pd.Series(y).value_counts()
keep = np.isin(y, vc[vc >= ccfg["min_project_size"]].index)
Xk, yk, sk = X[keep], y[keep], samples[keep]

le = LabelEncoder().fit(yk)
yi = le.transform(yk)
Xtr, Xte, ytr, yte, _, ste = train_test_split(
    Xk, yi, sk, test_size=0.2, random_state=SEED, stratify=yi)

scaler = StandardScaler().fit(Xtr)
Xtr_s = torch.tensor(scaler.transform(Xtr), dtype=torch.float32)
Xte_s = torch.tensor(scaler.transform(Xte), dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.long)

torch.manual_seed(SEED)
model = var.ProjectLinear(Xtr_s.shape[1], len(le.classes_)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=ccfg["lr"])
loss_fn = torch.nn.CrossEntropyLoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xtr_s, ytr_t),
    batch_size=ccfg["batch_size"], shuffle=True)

model.train()
for _ in range(ccfg["epochs"]):
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
        opt.step()

model.eval()
with torch.no_grad():
    logits = model(Xte_s.to(DEVICE)).cpu().numpy()
pred = logits.argmax(1)
top5 = np.argsort(-logits, axis=1)[:, :5]

guesses = pd.DataFrame({
    "sample": ste,
    "true_study": le.inverse_transform(yte),
    "predicted_study": le.inverse_transform(pred),
    "correct": pred == yte,
    "true_in_top5": [t in row for t, row in zip(yte, top5)],
})
print(f"held-out samples:            {len(guesses):,}")
print(f"exact-study accuracy:        {guesses['correct'].mean():.1%}")
print(f"true study in model top-5:   {guesses['true_in_top5'].mean():.1%}")
guesses.head(20)

### Which studies are easiest to recognise

In [ ]:
by_study = (guesses.groupby("true_study")
            .agg(n_heldout=("correct", "size"), recognised=("correct", "mean"))
            .sort_values("recognised", ascending=False))

print("most recognisable:")
print(by_study.head(10).round(3).to_string())
print("\nleast recognisable:")
print(by_study.tail(10).round(3).to_string())

## 5 — Figure + saved outputs

In [ ]:
fig = var.plot_project_classifier(cv, summary)
fig.savefig(FIGURES / "experiment_project_guessing.png", dpi=120, bbox_inches="tight")
guesses.to_csv(REPORTS / "experiment_heldout_guesses.csv", index=False)
print("wrote reports/figures/experiment_project_guessing.png")
print("wrote reports/experiment_heldout_guesses.csv")
fig

## What this shows

The model is given only relative genus abundances, yet it recovers the study of
origin far above the majority-class rate — a bare linear model is enough. Study
identity is a strong, low-dimensional, linearly accessible signal in this
compendium. Any disease classifier trained across these studies can exploit the
same signal, which is why the disease analyses in Phase 5 rely on
leave-one-study-out evaluation and within-study meta-analysis rather than
pooling samples across labs.